# Introduction to Machine Learning: Unsupervised Learning

**Instructor:** Daniel Acuna, Ph.D.
**Position:** Associate Professor of Computer Science
**Institution:** University of Colorado Boulder

---

Lab 1: Introduction to Unsupervised Learning and Data Exploration

---

In this lab, you will explore the foundations of unsupervised learning by working
with the **USArrests dataset**. This dataset contains crime statistics for 50 US
states across 3 features: Murder, Assault, UrbanPop (urban population percentage).

You will:
- Perform exploratory data analysis (EDA) on unlabeled data
- Understand why feature scaling is critical for distance-based methods
- Implement distance metrics (Euclidean and Manhattan)
- Build distance matrices and find similar/dissimilar observations
- Apply standardization and observe its impact on distances

These concepts form the foundation for clustering and dimensionality reduction
techniques you will learn in upcoming weeks.

## Setup (do not edit)

In [1]:
import pathlib
from typing import Tuple, Dict, List

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

RANDOM_STATE: int = 42
np.random.seed(RANDOM_STATE)

_DATA_PATH = pathlib.Path("us_arrests.csv")
if not _DATA_PATH.exists():
    raise FileNotFoundError(
        "us_arrests.csv is missing from the lab directory. Please run the download "
        "script (w1_download_datasets.py) or ask the TA for assistance."
    )

## 1. Load the Dataset *(10 points)*

Load the USArrests dataset from `us_arrests.csv` into a pandas DataFrame.
The first column contains state names and should be used as the index.

Store the following in the required variables:
- **`q1_shape`**: A tuple containing the shape of the DataFrame `(n_rows, n_cols)`
- **`q1_columns`**: A list of the column names

In [2]:
# Grade Cell: Question 1
#
# Task: Load the USArrests dataset and explore its structure
#
# Instructions:
# 1. Read the CSV file using pd.read_csv() with index_col=0 to use state names as index
# 2. Store the shape as a tuple in q1_shape
# 3. Store the column names as a list in q1_columns

# your code here
# raise NotImplementedError

df = pd.read_csv('us_arrests.csv', index_col=0)
print(df.index)
print(df.describe())
print(df.head())
print(df.index.size)
q1_shape = df.shape
q1_rows = list(df.index)
q1_columns = list(df.columns)
print(f'q1_shape: {q1_shape} | q1_columns: {q1_columns}')
print(f'rows: {q1_rows}')

Index(['Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California', 'Colorado',
       'Connecticut', 'Delaware', 'Florida', 'Georgia', 'Hawaii', 'Idaho',
       'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana',
       'Maine', 'Maryland', 'Massachusetts', 'Michigan', 'Minnesota',
       'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'Nevada',
       'New Hampshire', 'New Jersey', 'New Mexico', 'New York',
       'North Carolina', 'North Dakota', 'Ohio', 'Oklahoma', 'Oregon',
       'Pennsylvania', 'Rhode Island', 'South Carolina', 'South Dakota',
       'Tennessee', 'Texas', 'Utah', 'Vermont', 'Virginia', 'Washington',
       'West Virginia', 'Wisconsin', 'Wyoming'],
      dtype='object', name='rownames')
         Murder     Assault   UrbanPop
count  50.00000   50.000000  50.000000
mean    7.78800  170.760000  65.540000
std     4.35551   83.337661  14.474763
min     0.80000   45.000000  32.000000
25%     4.07500  109.000000  54.500000
50%     7.25000  159.000000  66.00

In [3]:
# If all tests pass (there might be hidden tests), you will earn 10 points
# Test Cell: Question 1
assert isinstance(q1_shape, tuple), (
    "q1_shape must be a tuple. Use df.shape which returns (rows, cols)."
)
assert len(q1_shape) == 2, (
    "q1_shape should have 2 elements (rows, cols). "
    "Make sure you're using .shape, not .shape[0]."
)
assert q1_shape[0] > 0 and q1_shape[1] > 0, (
    "Shape values must be positive. Is your CSV loading correctly?"
)
assert isinstance(q1_columns, list), (
    "q1_columns must be a list. Use df.columns.tolist() to convert."
)
assert len(q1_columns) == q1_shape[1], (
    "Number of columns in q1_columns should match q1_shape[1]."
)
print(f"Dataset shape: {q1_shape}")
print(f"Columns: {q1_columns}")

Dataset shape: (50, 3)
Columns: ['Murder', 'Assault', 'UrbanPop']


## 2. Compute Summary Statistics *(10 points)*

Compute the **mean** and **standard deviation** for each feature in the dataset.

Store the results as dictionaries:
- **`q2_means`**: A dictionary mapping feature names to their mean values (rounded to 2 decimals)
- **`q2_stds`**: A dictionary mapping feature names to their standard deviation values (rounded to 2 decimals)

Use `ddof=0` for standard deviation (population std) to match sklearn's StandardScaler.

In [4]:
# Grade Cell: Question 2
#
# Task: Compute summary statistics for each feature
#
# Instructions:
# 1. Calculate the mean of each column
# 2. Calculate the standard deviation of each column (use ddof=0)
# 3. Round values to 2 decimal places
# 4. Store as dictionaries with column names as keys

# your code here
# raise NotImplementedError


q2_means = {}
q2_stds = {}
for c in q1_columns:
    c_vector = df[c]
    c_mean = c_vector.mean()
    c_std = np.std(c_vector, ddof=0)
    q2_means[c] = round(c_mean, 2)
    q2_stds[c] = round(c_std, 2)

print(f'q2_means: {q2_means} \nq2_stds: {q2_stds}')

q2_means: {'Murder': 7.79, 'Assault': 170.76, 'UrbanPop': 65.54} 
q2_stds: {'Murder': 4.31, 'Assault': 82.5, 'UrbanPop': 14.33}


In [5]:
# If all tests pass (there might be hidden tests), you will earn 10 points
# Test Cell: Question 2
assert isinstance(q2_means, dict), (
    "q2_means must be a dictionary mapping feature names to mean values."
)
assert isinstance(q2_stds, dict), (
    "q2_stds must be a dictionary mapping feature names to std values."
)
assert len(q2_means) == 3, (
    "q2_means should have 3 entries, one for each feature."
)
assert len(q2_stds) == 3, (
    "q2_stds should have 3 entries, one for each feature."
)
assert all(isinstance(v, float) for v in q2_means.values()), (
    "All mean values should be floats. Use round() to ensure this."
)
assert all(v > 0 for v in q2_stds.values()), (
    "Standard deviations must be positive. Check your calculation."
)
print(f"Feature means: {q2_means}")
print(f"Feature stds: {q2_stds}")

Feature means: {'Murder': 7.79, 'Assault': 170.76, 'UrbanPop': 65.54}
Feature stds: {'Murder': 4.31, 'Assault': 82.5, 'UrbanPop': 14.33}


## 3. Understand Feature Scales *(10 points)*

One critical insight from unsupervised learning is that features with larger scales
can dominate distance calculations. To see this, compute the **range** (max - min)
for each feature.

Store the result as:
- **`q3_ranges`**: A dictionary mapping feature names to their range values (rounded to 2 decimals)

After computing, observe which feature has the largest range—this feature would
dominate Euclidean distances if we don't standardize!

In [6]:
# Grade Cell: Question 3
#
# Task: Compute the range (max - min) for each feature
#
# Instructions:
# 1. For each column, compute max() - min()
# 2. Round to 2 decimal places
# 3. Store as a dictionary

# your code here
# raise NotImplementedError

q3_ranges = {}
for c in q1_columns:
    c_range = df[c].max() - df[c].min()
    q3_ranges[c] = round(c_range, 2)

q3_ranges

{'Murder': 16.6, 'Assault': 292, 'UrbanPop': 59}

In [7]:
# If all tests pass (there might be hidden tests), you will earn 10 points
# Test Cell: Question 3
assert isinstance(q3_ranges, dict), (
    "q3_ranges must be a dictionary mapping feature names to range values."
)
assert len(q3_ranges) == 3, (
    "q3_ranges should have 3 entries, one for each feature."
)
assert all(v > 0 for v in q3_ranges.values()), (
    "Range values must be positive. Check your max - min calculation."
)
# Check that Assault has the largest range (it should dominate distances)
max_range_feature = max(q3_ranges, key=q3_ranges.get)
print(f"Feature ranges: {q3_ranges}")
print(f"Feature with largest range: {max_range_feature}")

Feature ranges: {'Murder': 16.6, 'Assault': 292, 'UrbanPop': 59}
Feature with largest range: Assault


## 4. Implement Euclidean Distance *(10 points)*

Distance metrics are the foundation of clustering algorithms. Implement a function
that computes the **Euclidean distance** between two vectors.

The Euclidean distance formula is:
$$d(\mathbf{x}, \mathbf{y}) = \sqrt{\sum_{j=1}^{p}(x_j - y_j)^2}$$

Create a function **`euclidean_distance(x, y)`** that:
- Takes two numpy arrays or pandas Series of the same length
- Returns the Euclidean distance as a float rounded to 3 decimal places

Then compute the distance between **Alabama** and **Alaska** (the first two states)
and store it in **`q4_distance_al_ak`**.

In [8]:
# Grade Cell: Question 4
#
# Task: Implement Euclidean distance function
#
# Instructions:
# 1. Implement the euclidean_distance function using the formula above
# 2. Use np.sqrt() and np.sum() for the calculation
# 3. Round the result to 3 decimal places
# 4. Compute distance between Alabama and Alaska

# your code here
# raise NotImplementedError


# a = np.array([[1, 2], [3, 4]])
# a = np.array([[1,],[3, 4]])
v_1 = np.array([1, 2, 3, 4])
print(f'np size: {v_1.size}')

print(df.head())

# get row by index string
# al_v = df.index["Alabama"]
# per https://pandas.pydata.org/docs/user_guide/indexing.html#different-choices-for-indexing 
# grab row by index string via .loc
al_v = df.loc["Alabama"]
ak_v = df.loc["Alaska"]
print(f'al:\n {al_v}')
print(f'ak:\n {ak_v}')
# ak_v

# al_v = df["Alabama"]
# ak_v = df["Alaska"]
print(f'vector size match: {al_v.size == ak_v.size}')

# instead of 2 for loops used zip 
# per https://docs.python.org/3/library/functions.html#zip
for vector_1_val, vector_2_val in zip(al_v, ak_v):
    print(f'vector_1_val: {vector_1_val} | vector_2_val: {vector_2_val}')

# subtract vectors via numpy
# via https://numpy.org/doc/stable/reference/generated/numpy.subtract.html#numpy-subtract
# np.subtract(al_v, ak_v) same as (al_v - ak_v) if numpy arrays
print('via numpy')
v_delta = np.subtract(al_v, ak_v)
print(v_delta)

# per https://pandas.pydata.org/docs/reference/api/pandas.Series.sub.html
# seems better via .subtract due to fill_value
print('via pandas')
print(al_v - ak_v)
print('via pandas .subtract')
print(al_v.subtract(ak_v, fill_value=0))

# per https://www.geeksforgeeks.org/python/calculate-the-euclidean-distance-using-numpy/
# using numpy array operations with python would satisfy the ask here. 
# removes negatives
v_delta_sq = v_delta**2
print('square v deltas')
print(v_delta_sq)

print(f'abs delta: {np.abs(v_delta)}')

v_delta_sum = np.sum(v_delta_sq)
print(f'v_delta_sum: {v_delta_sum}')
# reverses the square on the sum. Seems to help make the result easier to interpret
sqrt_sum = np.sqrt(v_delta_sum)
print(f'sqrt_sum: {sqrt_sum}')
print(f'sqrt_sum round: {round(sqrt_sum,3)}')



# Create a function euclidean_distance(x, y) that:

# Takes two numpy arrays or pandas Series of the same length
# Returns the Euclidean distance as a float rounded to 3 decimal places

# there is a function of the same name at https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.euclidean_distances.html
def euclidean_distance(x, y):
    if x.size != y.size:
        raise ValueError('vector size should match for substraction operation to work.')
    return float(round(np.sqrt(np.sum(np.subtract(x, y)**2)), 3))

# test via pandas
# ed_check_pd = euclidean_distance(pd.Series([1,2]), pd.Series([3,4])) # should work
# print(f'ed_check_pd: {ed_check_pd}') # works

assert euclidean_distance(pd.Series([1,2]), pd.Series([3,4])) == 2.828, 'should match'

# test via np array creations
# via https://numpy.org/doc/stable/user/basics.creation.html#d-array-creation-functions 
# euclidean_distance(np.arange(4), np.arange(3)) # should fail
ed_check = euclidean_distance(np.arange(3), np.arange(3)) # should work
print(f'ed_check: {ed_check}')

assert euclidean_distance(np.arange(3), np.arange(3)) == 0.0, 'should match'


# Then compute the distance between Alabama and Alaska (the first two states) and store it in q4_distance_al_ak.

q4_distance_al_ak = euclidean_distance(al_v, ak_v) # should work
print(f'q4_distance_al_ak: {q4_distance_al_ak}')

# test_x = np.array([1, 2])
# test_y = np.array([4, 5])
# test_dist = euclidean_distance(test_x, test_y)
# print(test_dist)

# print(euclidean_distance(np.arange(2), np.arange(2)))


np size: 4
            Murder  Assault  UrbanPop
rownames                             
Alabama       13.2      236        58
Alaska        10.0      263        48
Arizona        8.1      294        80
Arkansas       8.8      190        50
California     9.0      276        91
al:
 Murder       13.2
Assault     236.0
UrbanPop     58.0
Name: Alabama, dtype: float64
ak:
 Murder       10.0
Assault     263.0
UrbanPop     48.0
Name: Alaska, dtype: float64
vector size match: True
vector_1_val: 13.2 | vector_2_val: 10.0
vector_1_val: 236.0 | vector_2_val: 263.0
vector_1_val: 58.0 | vector_2_val: 48.0
via numpy
Murder       3.2
Assault    -27.0
UrbanPop    10.0
dtype: float64
via pandas
Murder       3.2
Assault    -27.0
UrbanPop    10.0
dtype: float64
via pandas .subtract
Murder       3.2
Assault    -27.0
UrbanPop    10.0
dtype: float64
square v deltas
Murder       10.24
Assault     729.00
UrbanPop    100.00
dtype: float64
abs delta: Murder       3.2
Assault     27.0
UrbanPop    10.0
dtype: flo

In [9]:
# If all tests pass (there might be hidden tests), you will earn 10 points
# Test Cell: Question 4
assert callable(euclidean_distance), (
    "euclidean_distance should be a function. Did you define it with 'def'?"
)
# Test with known values
test_x = np.array([1, 2, 3])
test_y = np.array([4, 5, 6])
test_dist = euclidean_distance(test_x, test_y)
assert isinstance(test_dist, float), (
    "euclidean_distance should return a float."
)
assert test_dist >= 0, (
    "Distance must be non-negative. Check your formula."
)
assert isinstance(q4_distance_al_ak, float), (
    "q4_distance_al_ak must be a float."
)
assert q4_distance_al_ak > 0, (
    "Distance between different states should be positive."
)
print(f"Test distance [1,2,3] to [4,5,6]: {test_dist}")
print(f"Distance Alabama to Alaska: {q4_distance_al_ak}")

Test distance [1,2,3] to [4,5,6]: 5.196
Distance Alabama to Alaska: 28.97


## 5. Implement Manhattan Distance *(10 points)*

Manhattan distance is an alternative to Euclidean distance that is more robust to
outliers. It sums the absolute differences rather than squared differences.

The Manhattan distance formula is:
$$d(\mathbf{x}, \mathbf{y}) = \sum_{j=1}^{p}|x_j - y_j|$$

Create a function **`manhattan_distance(x, y)`** that:
- Takes two numpy arrays or pandas Series of the same length
- Returns the Manhattan distance as a float rounded to 3 decimal places

Then compute the Manhattan distance between **Alabama** and **Alaska** and
store it in **`q5_manhattan_al_ak`**.

In [10]:
# Grade Cell: Question 5
#
# Task: Implement Manhattan distance function
#
# Instructions:
# 1. Implement the manhattan_distance function using the formula above
# 2. Use np.sum() and np.abs() for the calculation
# 3. Round the result to 3 decimal places
# 4. Compute distance between Alabama and Alaska

# your code here
# raise NotImplementedError


# Create a function manhattan_distance(x, y) that:

# Takes two numpy arrays or pandas Series of the same length
# Returns the Manhattan distance as a float rounded to 3 decimal places

def manhattan_distance(x, y):
    if x.size != y.size:
        raise ValueError('vector size should match for substraction operation to work.')
    return float(round(np.sum(np.abs(np.subtract(x, y))), 3))


# Then compute the Manhattan distance between Alabama and Alaska and store it in q5_manhattan_al_ak.
q5_manhattan_al_ak = manhattan_distance(al_v, ak_v) # should work
print(f'q5_manhattan_al_ak: {q5_manhattan_al_ak}')


q5_manhattan_al_ak: 40.2


In [11]:
# If all tests pass (there might be hidden tests), you will earn 10 points
# Test Cell: Question 5
assert callable(manhattan_distance), (
    "manhattan_distance should be a function. Did you define it with 'def'?"
)
# Test with known values
test_x = np.array([1, 2, 3])
test_y = np.array([4, 5, 6])
test_manhattan = manhattan_distance(test_x, test_y)
assert isinstance(test_manhattan, float), (
    "manhattan_distance should return a float."
)
assert test_manhattan >= 0, (
    "Distance must be non-negative. Check your formula."
)
# Manhattan should be >= Euclidean for the same points
assert test_manhattan >= euclidean_distance(test_x, test_y), (
    "Manhattan distance should be >= Euclidean distance for the same points."
)
assert isinstance(q5_manhattan_al_ak, float), (
    "q5_manhattan_al_ak must be a float."
)
print(f"Test Manhattan distance [1,2,3] to [4,5,6]: {test_manhattan}")
print(f"Manhattan distance Alabama to Alaska: {q5_manhattan_al_ak}")

Test Manhattan distance [1,2,3] to [4,5,6]: 9.0
Manhattan distance Alabama to Alaska: 40.2


## 6. Build a Distance Matrix *(15 points)*

A distance matrix shows the pairwise distances between all observations. This is
fundamental to many clustering algorithms.

Build a **Euclidean distance matrix** for all 50 states using the **unstandardized**
(original) data.

Store the result as:
- **`q6_dist_matrix`**: A 2D numpy array of shape (50, 50) containing pairwise distances

The diagonal should be 0 (distance from a state to itself), and the matrix should
be symmetric (distance from A to B equals distance from B to A).

**Hint:** You can use nested loops, or use `scipy.spatial.distance.pdist` and
`squareform` for efficiency.

In [12]:
# Grade Cell: Question 6
#
# Task: Build a pairwise Euclidean distance matrix
#
# Instructions:
# 1. Create a 50x50 matrix of zeros
# 2. For each pair of states, compute Euclidean distance
# 3. Store distances in the matrix (it should be symmetric)

# your code here
# raise NotImplementedError

from scipy.spatial.distance import pdist, cdist, squareform
from sklearn.utils.validation import check_symmetric

# 1. Create a 50x50 matrix of zeros
print(np.zeros((50, 50)))


# 2. For each pair of states, compute Euclidean distance

print(df.shape)

# calculate pairwise distance across all states
# where df is a ND array of form (Rows x Columns)
# pdist is used for a single set
# per https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.pdist.html
# 2nd param is metric and defaults to "euclidean" but had to confirm via source code.
# ref https://caam37830.github.io/book/08_geometry/distances.html#pdist 
pds_vector_dm = pdist(df, "euclidean")
print(pds_vector_dm.shape)

q6_dist_matrix = squareform(pds_vector_dm)
print(q6_dist_matrix.shape)
print('distance matrix')
print(q6_dist_matrix)

# from earlier
# q1_rows = list(df.index)
# q1_columns = list(df.columns)

# To make reading the matrix and interpreting much easier convert to a pandas dataframe 
# Wrap in DataFrame for easy reading
# ref https://kanoki.org/2019/12/27/how-to-calculate-distance-in-python-and-pandas-using-scipy-spatial-and-distance-functions/#euclidean-distance-metrics-using-scipy-spatial-pdist-function
# but after some trials I saw that its best rows and rows for interpretation
dist_df = pd.DataFrame(q6_dist_matrix, index=q1_rows, columns=q1_rows)
# dist_df = pd.DataFrame(q6_dist_matrix, index=q1_rows)
print('distance matrix dataframe')
display(dist_df)

# Observations:
# - noticed the euclidean distance between Alabama & Alaska differs slightly from earlier where
# manual earlier was 28.969639279770124
# and via matrix is 28.969639
# pdist docs does not explain any rounding or trimming.
# this at least confirms accuracy between the two approaches.

print('experiment 001 with distance matrix on 1 feature/column')
# experiement 001
# test another euclidean distance matrix but for only 1 feature/column instead of all as its was default above
# Only use the first two features/columns for the distance calculation
# df.iloc[:,0:1]
dist_vector = pdist(df.iloc[:, 0:1], metric='euclidean')
# dist_vector
dist_matrix = squareform(dist_vector)
dist_df_ex001 = pd.DataFrame(dist_matrix, index=q1_rows, columns=q1_rows)
display(dist_df_ex001)

# 3. Store distances in the matrix (it should be symmetric)
# how to check symmetry??
# https://scikit-learn.org/stable/modules/generated/sklearn.utils.validation.check_symmetric.html
# raises error if not symmetric per docs
check_symmetric(q6_dist_matrix)


[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
(50, 3)
(1225,)
(50, 50)
distance matrix
[[  0.          28.96963928  62.24154561 ... 156.34017398 183.4812252
   75.29913678]
 [ 28.96963928   0.          44.59383365 ... 182.27311925 210.89988146
  102.75329678]
 [ 62.24154561  44.59383365   0.         ... 216.92339662 241.4689421
  134.50163568]
 ...
 [156.34017398 182.27311925 216.92339662 ...   0.          39.02063557
   82.71765229]
 [183.4812252  210.89988146 241.4689421  ...  39.02063557   0.
  108.24804848]
 [ 75.29913678 102.75329678 134.50163568 ...  82.71765229 108.24804848
    0.        ]]
distance matrix dataframe


,Alabama,Alaska,Arizona,Arkansas,California,Colorado,Connecticut,Delaware,Florida,Georgia,...,South Dakota,Tennessee,Texas,Utah,Vermont,Virginia,Washington,West Virginia,Wisconsin,Wyoming
Alabama,0.000000,28.969639,62.241546,46.897335,52.025378,38.106299,127.808490,15.915087,101.438849,25.429117,...,150.855427,48.010416,41.343077,118.490506,190.107864,80.293773,92.685705,156.340174,183.481225,75.299137
Alaska,28.969639,0.000000,44.593834,73.037251,44.933284,66.222428,155.868181,34.897135,78.975692,53.877268,...,177.133961,75.869889,69.823277,146.694376,215.735579,108.056698,120.768373,182.273119,210.899881,102.753297
Arizona,62.241546,44.593834,0.000000,108.242736,21.114213,90.022442,184.087045,56.611306,41.644808,85.880673,...,210.967983,108.180451,93.113694,174.068981,250.708616,139.043734,149.220676,216.923397,241.468942,134.501636
Arkansas,46.897335,73.037251,108.242736,0.000000,95.273501,31.317886,84.612351,52.881093,148.217948,24.798387,...,104.240107,10.215674,32.190216,76.363342,143.288381,36.401786,50.764555,109.597491,138.070417,30.740852
California,52.025378,44.933284,21.114213,95.273501,0.000000,73.172467,166.686802,42.598239,60.356938,72.502138,...,195.558278,93.731745,75.892622,156.494856,235.608234,123.224389,132.325357,201.841250,224.488218,119.125312
Colorado,38.106299,66.222428,90.022442,31.317886,73.172467,0.000000,94.117799,34.583233,131.229760,21.523243,...,122.596126,25.398622,6.003332,84.155154,162.740560,50.292743,59.339784,129.053632,151.568763,46.628425
Connecticut,127.808490,155.868181,184.087045,84.612351,166.686802,94.117799,0.000000,128.124002,225.345091,103.386701,...,40.003125,80.659841,91.533382,10.440785,76.617296,48.363623,35.234784,47.861885,58.055921,53.872535
Delaware,15.915087,34.897135,56.611306,52.881093,42.598239,34.583233,128.124002,0.000000,97.791871,31.705678,...,154.393685,52.175569,38.460889,118.301691,194.200129,82.533387,93.024782,160.430795,185.126686,77.934652
Florida,101.438849,78.975692,41.644808,148.217948,60.356938,131.229760,225.345091,97.791871,0.000000,125.618470,...,251.715236,148.508720,134.027199,215.345861,291.285496,179.937795,190.470365,257.470561,282.637294,175.356665
Georgia,25.429117,53.877268,85.880673,24.798387,72.502138,21.523243,103.386701,31.705678,125.618470,0.000000,...,126.629223,23.401709,22.849289,94.247759,166.084436,55.796147,68.589795,132.203971,158.805038,51.111251


experiment 001 with distance matrix on 1 feature/column


,Alabama,Alaska,Arizona,Arkansas,California,Colorado,Connecticut,Delaware,Florida,Georgia,...,South Dakota,Tennessee,Texas,Utah,Vermont,Virginia,Washington,West Virginia,Wisconsin,Wyoming
Alabama,0.0,3.2,5.1,4.4,4.2,5.3,9.9,7.3,2.2,4.2,...,9.4,0.0,0.5,10.0,11.0,4.7,9.2,7.5,10.6,6.4
Alaska,3.2,0.0,1.9,1.2,1.0,2.1,6.7,4.1,5.4,7.4,...,6.2,3.2,2.7,6.8,7.8,1.5,6.0,4.3,7.4,3.2
Arizona,5.1,1.9,0.0,0.7,0.9,0.2,4.8,2.2,7.3,9.3,...,4.3,5.1,4.6,4.9,5.9,0.4,4.1,2.4,5.5,1.3
Arkansas,4.4,1.2,0.7,0.0,0.2,0.9,5.5,2.9,6.6,8.6,...,5.0,4.4,3.9,5.6,6.6,0.3,4.8,3.1,6.2,2.0
California,4.2,1.0,0.9,0.2,0.0,1.1,5.7,3.1,6.4,8.4,...,5.2,4.2,3.7,5.8,6.8,0.5,5.0,3.3,6.4,2.2
Colorado,5.3,2.1,0.2,0.9,1.1,0.0,4.6,2.0,7.5,9.5,...,4.1,5.3,4.8,4.7,5.7,0.6,3.9,2.2,5.3,1.1
Connecticut,9.9,6.7,4.8,5.5,5.7,4.6,0.0,2.6,12.1,14.1,...,0.5,9.9,9.4,0.1,1.1,5.2,0.7,2.4,0.7,3.5
Delaware,7.3,4.1,2.2,2.9,3.1,2.0,2.6,0.0,9.5,11.5,...,2.1,7.3,6.8,2.7,3.7,2.6,1.9,0.2,3.3,0.9
Florida,2.2,5.4,7.3,6.6,6.4,7.5,12.1,9.5,0.0,2.0,...,11.6,2.2,2.7,12.2,13.2,6.9,11.4,9.7,12.8,8.6
Georgia,4.2,7.4,9.3,8.6,8.4,9.5,14.1,11.5,2.0,0.0,...,13.6,4.2,4.7,14.2,15.2,8.9,13.4,11.7,14.8,10.6


array([[  0.        ,  28.96963928,  62.24154561, ..., 156.34017398,
        183.4812252 ,  75.29913678],
       [ 28.96963928,   0.        ,  44.59383365, ..., 182.27311925,
        210.89988146, 102.75329678],
       [ 62.24154561,  44.59383365,   0.        , ..., 216.92339662,
        241.4689421 , 134.50163568],
       ...,
       [156.34017398, 182.27311925, 216.92339662, ...,   0.        ,
         39.02063557,  82.71765229],
       [183.4812252 , 210.89988146, 241.4689421 , ...,  39.02063557,
          0.        , 108.24804848],
       [ 75.29913678, 102.75329678, 134.50163568, ...,  82.71765229,
        108.24804848,   0.        ]])

In [13]:
# If all tests pass (there might be hidden tests), you will earn 15 points
# Test Cell: Question 6
assert isinstance(q6_dist_matrix, np.ndarray), (
    "q6_dist_matrix must be a numpy array."
)
assert q6_dist_matrix.shape == (50, 50), (
    f"Distance matrix should be 50x50, got {q6_dist_matrix.shape}. "
    "Make sure you're computing distances for all pairs of states."
)
# Check diagonal is zeros (distance to self)
assert np.allclose(np.diag(q6_dist_matrix), 0), (
    "Diagonal elements should be 0 (distance from a state to itself)."
)
# Check symmetry
assert np.allclose(q6_dist_matrix, q6_dist_matrix.T), (
    "Distance matrix should be symmetric. d(A,B) = d(B,A)."
)
# Check all values are non-negative
assert (q6_dist_matrix >= 0).all(), (
    "All distances must be non-negative."
)
print(f"Distance matrix shape: {q6_dist_matrix.shape}")
print(f"Diagonal (should be zeros): {np.diag(q6_dist_matrix)[:5]}")
print(f"Max distance: {q6_dist_matrix.max():.2f}")

Distance matrix shape: (50, 50)
Diagonal (should be zeros): [0. 0. 0. 0. 0.]
Max distance: 293.57


## 7. Standardize the Data *(10 points)*

As we learned in the lectures, **standardization is critical** before computing
distances. Without it, features with larger scales dominate the distance calculation.

Apply z-score standardization to the data:
$$z_j = \frac{x_j - \mu_j}{\sigma_j}$$

Use `sklearn.preprocessing.StandardScaler` to standardize the data.

Store the results as:
- **`q7_scaled_data`**: A pandas DataFrame with the standardized values (same index and columns as original)
- **`q7_scaled_means`**: A dictionary of the means of the scaled data (should all be ~0)
- **`q7_scaled_stds`**: A dictionary of the stds of the scaled data (should all be ~1)

In [14]:
# Grade Cell: Question 7
#
# Task: Standardize the data using StandardScaler
#
# Instructions:
# 1. Create a StandardScaler instance
# 2. Fit and transform the data
# 3. Convert back to DataFrame with original index and columns
# 4. Compute means and stds of the scaled data (should be ~0 and ~1)

# your code here
# raise NotImplementedError

# Attribution: most of the code below was taken in pieces from various online sources including genAI 
# via google browser ai mode which sometimes cites sources. 
# It was honestly a trial and error approach here so I could not find a reference that covered this well except 
# via https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html#sklearn.preprocessing.StandardScaler
# then https://scikit-learn.org/stable/modules/preprocessing.html#standardization-or-mean-removal-and-variance-scaling
# but part on applying StandardScaler and transformation to an unsupervise dataset (without splitting) 
# was nowhere to be found so I had to rely on AI unfortunately.
# The closest ref was https://amueller.github.io/aml/01-ml-workflow/03-preprocessing.html#ways-to-scale-data
# which helped me connect the dots of using the original 2D array as the "training set" for this. The part that 
# clicked suprisingly was the term "scaling" which was also found in the sklearn docs above but was not explicitly 
# synonymous but seems like it is to standarizing.


# 1. Create a StandardScaler instance

# data = [[0, 0], [0, 0], [1, 1], [1, 1]]
print(df)
scaler = StandardScaler()


# 2. Fit and transform the data

# print(scaler.fit(data))
# StandardScaler()
# print(scaler.mean_)

# Fit the scaler on the training data and transform the training data
# X_train_scaled = scaler.fit_transform(X_train)
orig_scaled = scaler.fit_transform(df)

# Transform the test data using the same scaler instance
# X_test_scaled = scaler.transform(X_test)


print("scalled original data:\n", orig_scaled)


# 3. Convert back to DataFrame with original index and columns

# A pandas DataFrame with the standardized values (same index and columns as original)
print(type(orig_scaled))# is numpy.ndarray 
print(orig_scaled.shape)# (50, 3)
assert orig_scaled.shape == (50, 3), "not the expected tuple"
q7_scaled_data = dist_df_ex001 = pd.DataFrame(orig_scaled, index=q1_rows, columns=q1_columns)

print(f"q7_scaled_data: \n{q7_scaled_data}")



# 4. Compute means and stds of the scaled data (should be ~0 and ~1)

# Below mean & std seems a bit odd. Does work but feels like the StandardScaler should provide this
# rather than having to run .std() again. Its confusing but could not see anything obvious in the docs.
# I'm not sure this is the correct implementation of this request but does
# satisdy the mean <0 and std = 1. BUT...
# per https://scikit-learn.org/stable/modules/preprocessing.html#standardization-or-mean-removal-and-variance-scaling
# I took it from there and seems to justify it, just not intuitive.

# Retrieve the parameters used for scaling
original_mean = scaler.mean_
original_std = scaler.scale_

print(f"Original Mean: {original_mean}")
print(f"Original Std Dev: {original_std}")
# print(f"Scaled Mean: {orig_scaled.mean_}")# not a numpy array so fails


# Verify the transformation (mean should be ~0, std should be 1)
# mean on numpy array via https://numpy.org/doc/stable/reference/generated/numpy.mean.html#numpy.mean
# axis 0 moves along each column vertically where axis index refers to shape (rows, columns)
scaled_means = orig_scaled.mean(axis=0)
scaled_stds = orig_scaled.std(axis=0)
print(f"scaled Mean: {scaled_means}")
print(f"scaled Std Dev: {scaled_stds}")

# Rounding to 10 decimal places
print(f"scaled Mean rounded: {np.round(orig_scaled.mean(axis=0), 10)}")

# A dictionary of the means of the scaled data (should all be ~0)
print('scaled_means')
print(scaled_means.shape)
scales_means_df = pd.DataFrame(scaled_means, index=q1_columns)
print(scales_means_df)
print(type(scales_means_df))
print(f"scaled mean of 1 feature: {q7_scaled_data['Murder'].mean()}") # confirms value matches what scales_means_df shows
# print(scales_means_df.iloc[0:1])
print(scaled_means[0])
assert scaled_means[0] == q7_scaled_data['Murder'].mean(), 'first feature mean not matching between using mean(axis=0) or not'
print(scales_means_df.to_dict().get(0))

q7_scaled_means = scales_means_df.to_dict().get(0)
assert isinstance(list(q7_scaled_means.values()), list), (
    "means values is a list"
)


# A dictionary of the stds of the scaled data (should all be ~1)
print('scaled_stds')
print(scaled_stds)
scaled_stds_df = pd.DataFrame(scaled_stds, index=q1_columns)
print(scaled_stds_df)

q7_scaled_stds = scaled_stds_df.to_dict().get(0)
assert isinstance(list(q7_scaled_stds.values()), list), (
    "stds values is a list"
)



                Murder  Assault  UrbanPop
rownames                                 
Alabama           13.2      236        58
Alaska            10.0      263        48
Arizona            8.1      294        80
Arkansas           8.8      190        50
California         9.0      276        91
Colorado           7.9      204        78
Connecticut        3.3      110        77
Delaware           5.9      238        72
Florida           15.4      335        80
Georgia           17.4      211        60
Hawaii             5.3       46        83
Idaho              2.6      120        54
Illinois          10.4      249        83
Indiana            7.2      113        65
Iowa               2.2       56        57
Kansas             6.0      115        66
Kentucky           9.7      109        52
Louisiana         15.4      249        66
Maine              2.1       83        51
Maryland          11.3      300        67
Massachusetts      4.4      149        85
Michigan          12.1      255   

In [15]:
# If all tests pass (there might be hidden tests), you will earn 10 points
# Test Cell: Question 7
assert isinstance(q7_scaled_data, pd.DataFrame), (
    "q7_scaled_data must be a pandas DataFrame."
)
assert q7_scaled_data.shape == df.shape, (
    "Scaled data should have same shape as original."
)
assert list(q7_scaled_data.columns) == list(df.columns), (
    "Scaled data should have same column names as original."
)
assert isinstance(q7_scaled_means, dict), (
    "q7_scaled_means must be a dictionary."
)
assert isinstance(q7_scaled_stds, dict), (
    "q7_scaled_stds must be a dictionary."
)
# Check that means are approximately 0
all_means_near_zero = all(abs(v) < 0.01 for v in q7_scaled_means.values())
assert all_means_near_zero, (
    "After standardization, all feature means should be ~0. "
    "Did you use StandardScaler correctly?"
)
# Check that stds are approximately 1
all_stds_near_one = all(abs(v - 1.0) < 0.01 for v in q7_scaled_stds.values())
assert all_stds_near_one, (
    "After standardization, all feature stds should be ~1. "
    "Did you use StandardScaler correctly?"
)
print(f"Scaled means (should be ~0): {q7_scaled_means}")
print(f"Scaled stds (should be ~1): {q7_scaled_stds}")

Scaled means (should be ~0): {'Murder': -7.105427357601002e-17, 'Assault': 1.3877787807814457e-16, 'UrbanPop': -4.39648317751562e-16}
Scaled stds (should be ~1): {'Murder': 1.0, 'Assault': 1.0, 'UrbanPop': 1.0}


## 8. Distance Matrix on Standardized Data *(10 points)*

Now build a distance matrix using the **standardized** data and compare it to
the unstandardized version.

Store the results as:
- **`q8_scaled_dist_matrix`**: A 2D numpy array of shape (50, 50) with pairwise distances on scaled data
- **`q8_al_ak_scaled`**: The Euclidean distance between Alabama and Alaska using scaled data (rounded to 3 decimals)

Compare `q8_al_ak_scaled` to `q4_distance_al_ak` to see how standardization changes distances.

In [16]:
# Grade Cell: Question 8
#
# Task: Build distance matrix on standardized data
#
# Instructions:
# 1. Compute pairwise Euclidean distances on q7_scaled_data
# 2. Extract the distance between Alabama and Alaska

# your code here
# raise NotImplementedError



# 1. Compute pairwise Euclidean distances on q7_scaled_data

from scipy.spatial.distance import pdist, cdist, squareform

print(df.shape)
print(q7_scaled_data.shape)
print(f'orig head: \n{df.head(2)}')
print(f'scaled head: \n{q7_scaled_data.head(2)}')

# pdist -> pairwise distances -> condensed distance matrix
scaled_pwd_cdm = pdist(q7_scaled_data, "euclidean")
# print(pds_vector_dm.shape)

q8_scaled_dist_matrix = squareform(scaled_pwd_cdm)
# print(q6_dist_matrix.shape)

# apply column names to a new Dataframe for easier interpretation
scaled_dm_df = pd.DataFrame(q8_scaled_dist_matrix, index=q1_rows, columns=q1_rows)

# get 1st row from original dm to compare visually
print('first row')
# display(dist_df.iloc[[0]]) # works but odd
# display(dist_df.iloc[:1]) # works
display(dist_df.head(1))


print(q8_scaled_dist_matrix)
print('scaled distance matrix')
display(scaled_dm_df)



# 2. Extract the distance between Alabama and Alaska

al_v_scaled = q7_scaled_data.loc["Alabama"]
ak_v_scaled = q7_scaled_data.loc["Alaska"]
print(f'al_v_scaled: \n{al_v_scaled}')
print(f'ak_v_scaled: \n{ak_v_scaled}')

q8_al_ak_scaled = euclidean_distance(al_v_scaled, ak_v_scaled) # should work
print(f'q4_distance_al_ak: {q4_distance_al_ak}')
print(f'q8_al_ak_scaled: {q8_al_ak_scaled}')

orig_scaled_distance_diff_abs = np.abs(q8_al_ak_scaled - q4_distance_al_ak)
print(f'orig_scaled_distance_diff_abs: {orig_scaled_distance_diff_abs}')


(50, 3)
(50, 3)
orig head: 
          Murder  Assault  UrbanPop
rownames                           
Alabama     13.2      236        58
Alaska      10.0      263        48
scaled head: 
           Murder   Assault  UrbanPop
Alabama  1.255179  0.790787 -0.526195
Alaska   0.513019  1.118060 -1.224067
first row


,Alabama,Alaska,Arizona,Arkansas,California,Colorado,Connecticut,Delaware,Florida,Georgia,...,South Dakota,Tennessee,Texas,Utah,Vermont,Virginia,Washington,West Virginia,Wisconsin,Wyoming
Alabama,0.0,28.969639,62.241546,46.897335,52.025378,38.106299,127.80849,15.915087,101.438849,25.429117,...,150.855427,48.010416,41.343077,118.490506,190.107864,80.293773,92.685705,156.340174,183.481225,75.299137


[[0.         1.07001609 2.06167651 ... 2.88333922 3.35794339 1.74617746]
 [1.07001609 0.         2.30705563 ... 2.5011495  3.31704529 1.66754619]
 [2.06167651 2.30705563 0.         ... 3.89390349 3.33394055 2.15358602]
 ...
 [2.88333922 2.5011495  3.89390349 ... 0.         2.04511937 1.77571801]
 [3.35794339 3.31704529 3.33394055 ... 2.04511937 0.         1.68460326]
 [1.74617746 1.66754619 2.15358602 ... 1.77571801 1.68460326 0.        ]]
scaled distance matrix


,Alabama,Alaska,Arizona,Arkansas,California,Colorado,Connecticut,Delaware,Florida,Georgia,...,South Dakota,Tennessee,Texas,Utah,Vermont,Virginia,Washington,West Virginia,Wisconsin,Wyoming
Alabama,0.000000,1.070016,2.061677,1.289941,2.547081,1.899866,3.059838,1.954888,2.014333,1.029637,...,2.980214,0.585988,1.597069,3.116590,3.872164,1.500091,2.620154,2.883339,3.357943,1.746177
Alaska,1.070016,0.000000,2.307056,0.938026,3.013919,2.265364,3.154332,1.949689,2.705047,2.010995,...,2.591228,1.402336,2.438038,3.237096,3.363163,1.702633,2.650676,2.501149,3.317045,1.667546
Arizona,2.061677,2.307056,0.000000,2.449225,0.824908,1.100778,2.501476,1.016262,1.764486,2.759074,...,3.649262,2.279840,1.552071,2.395775,4.688782,2.052832,2.098735,3.893903,3.333941,2.153586
Arkansas,1.289941,0.938026,2.449225,0.000000,3.045601,1.972471,2.473428,1.774283,3.132946,2.128397,...,1.748031,1.198515,2.284543,2.605759,2.623659,0.998878,2.028099,1.688731,2.464153,0.908705
California,2.547081,3.013919,0.824908,3.045601,0.000000,1.284449,2.598232,1.577096,1.817677,3.016029,...,4.130841,2.659653,1.467011,2.444239,5.203674,2.438733,2.333248,4.397918,3.543088,2.623683
Colorado,1.899866,2.265364,1.100778,1.972471,1.284449,0.000000,1.562456,0.748552,2.359340,2.537645,...,2.872918,1.818437,1.122546,1.498125,3.953301,1.205687,1.204713,3.144963,2.358444,1.383729
Connecticut,3.059838,3.154332,2.501476,2.473428,2.598232,1.562456,0.000000,1.700755,3.918817,3.687834,...,2.255041,2.782756,2.452209,0.243028,3.239153,1.649218,0.533162,2.732403,1.045472,1.564790
Delaware,1.954888,1.949689,1.016262,1.774283,1.577096,0.748552,1.700755,0.000000,2.559020,2.814615,...,2.679952,2.014153,1.732066,1.658187,3.719235,1.321370,1.212349,2.987870,2.406149,1.271218
Florida,2.014333,2.705047,1.764486,3.132946,1.817677,2.359340,3.918817,2.559020,0.000000,2.102939,...,4.723700,2.362835,1.740771,3.846757,5.717975,2.945503,3.540203,4.767267,4.631561,3.220964
Georgia,1.029637,2.010995,2.759074,2.128397,3.016029,2.537645,3.687834,2.814615,2.102939,0.000000,...,3.652445,1.015596,1.775105,3.743107,4.488804,2.179203,3.334888,3.463236,3.952864,2.532010


al_v_scaled: 
Murder      1.255179
Assault     0.790787
UrbanPop   -0.526195
Name: Alabama, dtype: float64
ak_v_scaled: 
Murder      0.513019
Assault     1.118060
UrbanPop   -1.224067
Name: Alaska, dtype: float64
q4_distance_al_ak: 28.97
q8_al_ak_scaled: 1.07
orig_scaled_distance_diff_abs: 27.9


In [17]:
# If all tests pass (there might be hidden tests), you will earn 10 points
# Test Cell: Question 8
assert isinstance(q8_scaled_dist_matrix, np.ndarray), (
    "q8_scaled_dist_matrix must be a numpy array."
)
assert q8_scaled_dist_matrix.shape == (50, 50), (
    f"Distance matrix should be 50x50, got {q8_scaled_dist_matrix.shape}."
)
assert np.allclose(np.diag(q8_scaled_dist_matrix), 0), (
    "Diagonal elements should be 0."
)
assert np.allclose(q8_scaled_dist_matrix, q8_scaled_dist_matrix.T), (
    "Distance matrix should be symmetric."
)
assert isinstance(q8_al_ak_scaled, float), (
    "q8_al_ak_scaled must be a float."
)
print(f"Unstandardized Alabama-Alaska distance: {q4_distance_al_ak}")
print(f"Standardized Alabama-Alaska distance: {q8_al_ak_scaled}")
print(f"Ratio (unstandardized / standardized): {q4_distance_al_ak / q8_al_ak_scaled:.2f}")

Unstandardized Alabama-Alaska distance: 28.97
Standardized Alabama-Alaska distance: 1.07
Ratio (unstandardized / standardized): 27.07


## 9. Find the Most Similar States *(10 points)*

Using the **standardized distance matrix** from Question 8, find the pair of states
that are **most similar** (smallest non-zero distance).

Store the results as:
- **`q9_most_similar_pair`**: A tuple of two state names (strings) representing the closest pair
- **`q9_min_distance`**: The distance between them (rounded to 3 decimals)

**Hint:** Set diagonal to infinity before finding the minimum, since diagonal is 0.

In [18]:
# Grade Cell: Question 9
#
# Task: Find the two most similar states based on standardized distances
#
# Instructions:
# 1. Copy the distance matrix and set diagonal to infinity
# 2. Find the indices of the minimum value
# 3. Get the state names corresponding to those indices

# your code here
# raise NotImplementedError


# find the pair of states that are most similar (smallest non-zero distance).
# This seems to be more complex at closer look.
# initial ideas involve:
# 1. recall how a distance matrix was made from observations(rows) and their dimensions/features. 
# It determined the pairwise distance between these dimensions but its possible on a single dimension. 
# Now attempt to apply it to a vector of a observations to a single dimension. Where states are observations and
# the distance values are 1 dimension. This would be a (50,1) vector. Then apply euclidean distance matrix again.
# not positive this will work but its an idea.
# 2. apply a sort by asc to a vector. Noticed the observations have the same values across dimensions in a matrix.
# where these are (states, states). Hence why (state, state) is 0 as is itself. Then getting the first 2 as a pair 
# since they are the smallest and closest. Not sure this assumption is true to all distance matrix operations but 
# it is fair for this scenario/domain which is valid on unsupervised learning problems. 


# A tuple of two state names (strings) representing the closest pair
print(q8_scaled_dist_matrix.ndim) #2 dimensions
print(type(q8_scaled_dist_matrix))

# 1. Copy the distance matrix and set diagonal to infinity
# found https://numpy.org/doc/stable/reference/generated/numpy.fill_diagonal.html
# and it does seem to modify the original in place so that explains the need to clone it first
# works with a ND array but with Dataframe.values() seems to work too.
# Could not find any solid docs on whether infinity in python is bad for computation. Its not common in 
# programming enough to be familiar so this is odd. Though in math seems common enough, hence the ask here to
# ignore the self-distance values (0) in the diagonal. It does pose various other nuances so best to use carefully.

# Set the diagonal to infinity
scaled_dm_cp = q8_scaled_dist_matrix.copy()
np.fill_diagonal(scaled_dm_cp, np.inf)
print(scaled_dm_cp)


# 2. Find the indices of the minimum value

# 1. Get the flat index of the minimum value
# ref https://numpy.org/doc/stable/reference/generated/numpy.argmin.html#numpy-argmin
# argmin return min indices 
# argmin flattens the array and returns an index unless the axis=int is passed.
# so need to convert it back with unravel_index
# NOTE: not sure what would happen if a non-2ndim array is passed downstream in this interpretation. 
# Maybe Some future exploration...
# ran into in the docs but did not try...a.flat[np.argmin(a)]..but could apply to ND arrays indices...
flat_index = np.argmin(scaled_dm_cp)

# 2. Convert to (row, col) coordinates
# row_idx, col_idx = np.unravel_index(flat_index, scaled_dm_cp.shape)
min_indices_tuple = np.unravel_index(flat_index, scaled_dm_cp.shape)
print(f'min_indices_tuple: {min_indices_tuple}')

# apply indices to dataframe having labels & numpy ND array to get value
print(scaled_dm_cp[min_indices_tuple])# return a single value but not sure how exactly...need to investigate
# print(scaled_dm_df[min_indices_tuple])#not work but desired for interpretation
row_idx, col_idx = min_indices_tuple
# on numpy array
min_dist_value = scaled_dm_cp[row_idx, col_idx]
print(f'value from numpy array: {min_dist_value}')
# on dataframe
row_label = scaled_dm_df.index[row_idx]
col_label = scaled_dm_df.columns[col_idx]

print(f"The closest pair is {row_label} and {col_label} with a distance of {min_dist_value}")

# q9_min_distance: The distance between them (rounded to 3 decimals)
q9_min_distance = float(round(min_dist_value, 3))
print(f'q9_min_distance: {q9_min_distance}')

# 3. Get the state names corresponding to those indices

q9_most_similar_pair = row_label, col_label
print(f'q9_most_similar_pair: {q9_most_similar_pair} | is tuple {type(q9_most_similar_pair)}')

# distance matrix to compare
print('Iowa vector column')
display(scaled_dm_df['Iowa'])


2
<class 'numpy.ndarray'>
[[       inf 1.07001609 2.06167651 ... 2.88333922 3.35794339 1.74617746]
 [1.07001609        inf 2.30705563 ... 2.5011495  3.31704529 1.66754619]
 [2.06167651 2.30705563        inf ... 3.89390349 3.33394055 2.15358602]
 ...
 [2.88333922 2.5011495  3.89390349 ...        inf 2.04511937 1.77571801]
 [3.35794339 3.31704529 3.33394055 ... 2.04511937        inf 1.68460326]
 [1.74617746 1.66754619 2.15358602 ... 1.77571801 1.68460326        inf]]
min_indices_tuple: (14, 28)
0.07453229808709586
value from numpy array: 0.07453229808709586
The closest pair is Iowa and New Hampshire with a distance of 0.07453229808709586
q9_min_distance: 0.075
q9_most_similar_pair: ('Iowa', 'New Hampshire') | is tuple <class 'tuple'>
Iowa vector column


Alabama           3.357633
Alaska            3.156352
Arizona           3.573668
Arkansas          2.284702
California        3.902348
Colorado          2.667134
Connecticut       1.562566
Delaware          2.588219
Florida           4.835833
Georgia           4.000144
Hawaii            1.955478
Idaho             0.808849
Illinois          3.518783
Indiana           1.460747
Iowa              0.000000
Kansas            1.297169
Kentucky          1.886826
Louisiana         3.903779
Maine             0.531953
Maryland          3.699805
Massachusetts     2.312867
Michigan          3.535210
Minnesota         0.667495
Mississippi       4.155752
Missouri          2.344589
Montana           1.125766
Nebraska          0.818448
Nevada            3.718657
New Hampshire     0.074532
New Jersey        2.828481
New Mexico        3.616713
New York          3.757196
North Carolina    4.310050
North Dakota      0.972767
Ohio              1.891777
Oklahoma          1.719490
Oregon            1.561365
P

In [19]:
# If all tests pass (there might be hidden tests), you will earn 10 points
# Test Cell: Question 9
assert isinstance(q9_most_similar_pair, tuple), (
    "q9_most_similar_pair must be a tuple of two state names."
)
assert len(q9_most_similar_pair) == 2, (
    "q9_most_similar_pair should contain exactly 2 state names."
)
assert all(isinstance(s, str) for s in q9_most_similar_pair), (
    "Both elements in q9_most_similar_pair should be strings (state names)."
)
assert q9_most_similar_pair[0] != q9_most_similar_pair[1], (
    "The two states should be different. Did you exclude the diagonal?"
)
assert isinstance(q9_min_distance, float), (
    "q9_min_distance must be a float."
)
assert q9_min_distance > 0, (
    "Minimum distance should be positive (not self-distance)."
)
print(f"Most similar states: {q9_most_similar_pair}")
print(f"Distance between them: {q9_min_distance}")

Most similar states: ('Iowa', 'New Hampshire')
Distance between them: 0.075


## 10. Find k-Nearest Neighbors *(5 points)*

For a given anchor state, find its **k nearest neighbors** based on standardized
Euclidean distance.

Implement a function **`find_k_nearest(state_name: str, k: int) -> List[str]`** that:
- Takes a state name and number of neighbors k
- Returns a list of the k nearest state names (excluding the anchor state itself)

Then find the **3 nearest neighbors of California** and store them in **`q10_ca_neighbors`**.

In [20]:
# Grade Cell: Question 10
#
# Task: Implement k-nearest neighbors lookup
#
# Instructions:
# 1. Get the row of distances for the given state
# 2. Sort indices by distance (excluding the state itself)
# 3. Return the top k state names

# your code here
# raise NotImplementedError

from sklearn.neighbors import KNeighborsRegressor

# This problem was a bit complex to figure out without AI. I used:
# - Google AI mode
# - Microsoft Copilot via Claude Sonnet 4.5 Agent mode. 
# How I used AI:
# - it was mostly to get un-stuck in parts that I just could not wrap my head around 
# - specially on how to tie KNN to Distance Matrix. This felt abstract to me. 
# - there was at least 50/50 my own / ai on this. 
# Tried to spend time understanding the whys and docs.

# ref https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html#sklearn.neighbors.KNeighborsRegressor
def find_k_nearest(state_name: str, k: int) -> List[str]:
#   Need to set k+1 because final output returns only k-1 excluding the state itself.
#   This accounts for this, since we're asking for k nearest neighbor states.
    k_states = k+1
# 
#   precomputed indicates a pre-existing distance matrix is used over re-computing but not positive...
#     neigh = KNeighborsRegressor(n_neighbors=k, metric='precomputed') # see note above 
    neigh = KNeighborsRegressor(n_neighbors=k_states, metric='precomputed')
#   Determine datasets to apply KNN on from scaled distance matrix.
#   Manually split over train_test_split because this is not supervised learning and 
#   we are not applying validation via MSE/RMSE
#
#   1. Get the row of distances for the given state
#   prep X as predictors in ND array where dimensions are other states but the target
#   prep y as response/target as 1D array/vector/pd.Series where dimension is only the one state requested for prediction
    X_nd_array = scaled_dm_df.drop(columns=[state_name])
    y_vector = scaled_dm_df[state_name]
    print(f'y_vector: shape {y_vector.shape} type {type(y_vector)} \n{y_vector}')
    # need to convert pd.Series to numpy array and reshape to 2D for sklearn
#     print(f'y_vector reshaped: \n{y_vector.reshape(1,-1)}')
    y_vector_reshaped = y_vector.values.reshape(1, -1)
    print(f'y_vector reshaped: \n{y_vector_reshaped}')

#   error with fit as it expects a square matrix (50,50) but X_nd_array was (50,49)
#     neigh.fit(X_nd_array, y_vector)
# Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.
    neigh.fit(scaled_dm_df, y_vector)
#   what to predict was a bit odd to understand but I figured the same distance matrix
#   KNN predictions on the/relative target state
#     y_pred = neigh.predict(scaled_dm_df)

#   2. Sort indices by distance (excluding the state itself)
#   Got Error: "Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample."
#     print(neigh.kneighbors(y_vector))
    print(neigh.kneighbors(y_vector_reshaped))

#   Return only indices 
#   need distances too so commented out
#     flat_index = neigh.kneighbors(y_vector_reshaped, return_distance=False)
#     print(f'flat_index: {flat_index}')
    
#     # Get the indices (shape: (1, k)) and flatten to 1D array
#     neighbor_indices = flat_index[0]  # Now shape: (k,)
#     print(f'neighbor_indices: {neighbor_indices}')

#     # Get the state names using these indices
#     nearest_states = scaled_dm_df.index[neighbor_indices].tolist()
#     print(f'nearest_states: {nearest_states}')
#   Return only indices END
    
#   TODO: how to sort by distance for k neighbors?????
#   Seems like neigh.kneighbors(y_vector_reshaped) return distances and indices sorted asc
    
    
#   Returns a list of the k nearest state names (excluding the anchor state itself)
#   3. Return the top k state names
    k_distances_list, k_indices_list = neigh.kneighbors(y_vector_reshaped)
    k_distances = k_distances_list[0]
    k_indices = k_indices_list[0]
    map_dict = {}
    for d, i in zip(k_distances, k_indices):
        print(d)
        if d == 0:
            continue
        map_dict[i] = d
    
    neighbor_indices = list(map_dict.keys())
    print(f'neighbor_indices: {neighbor_indices}')
    
    # Get the state names using these indices
    nearest_states = scaled_dm_df.index[neighbor_indices].tolist()
    print(f'nearest_states: {nearest_states}')
    
    return nearest_states

# find_k_nearest('Iowa', 3)

q10_ca_neighbors = find_k_nearest('California', 3)


y_vector: shape (50,) type <class 'pandas.core.series.Series'> 
Alabama           2.547081
Alaska            3.013919
Arizona           0.824908
Arkansas          3.045601
California        0.000000
Colorado          1.284449
Connecticut       2.598232
Delaware          1.577096
Florida           1.817677
Georgia           3.016029
Hawaii            2.969903
Idaho             3.527904
Illinois          0.724037
Indiana           2.714807
Iowa              3.902348
Kansas            2.708581
Kentucky          3.395814
Louisiana         2.313919
Maine             3.978198
Maryland          1.781695
Massachusetts     1.919177
Michigan          1.410394
Minnesota         3.360531
Mississippi       3.675916
Missouri          1.886487
Montana           3.407974
Nebraska          3.119670
Nevada            1.059460
New Hampshire     3.946335
New Jersey        1.472555
New Mexico        1.571466
New York          0.655803
North Carolina    3.422393
North Dakota      4.713295
Ohio              

In [21]:
# If all tests pass (there might be hidden tests), you will earn 5 points
# Test Cell: Question 10
assert callable(find_k_nearest), (
    "find_k_nearest should be a function."
)
assert isinstance(q10_ca_neighbors, list), (
    "q10_ca_neighbors must be a list of state names."
)
assert len(q10_ca_neighbors) == 3, (
    "q10_ca_neighbors should contain exactly 3 neighbors."
)
assert all(isinstance(s, str) for s in q10_ca_neighbors), (
    "All neighbors should be strings (state names)."
)
assert "California" not in q10_ca_neighbors, (
    "California should not be in its own neighbor list."
)
# Test the function with another state
test_neighbors = find_k_nearest("Texas", 2)
assert len(test_neighbors) == 2, (
    "find_k_nearest(Texas, 2) should return 2 neighbors."
)
assert "Texas" not in test_neighbors, (
    "A state should not appear in its own neighbor list."
)
print(f"California's 3 nearest neighbors: {q10_ca_neighbors}")
print(f"Texas's 2 nearest neighbors: {test_neighbors}")

y_vector: shape (50,) type <class 'pandas.core.series.Series'> 
Alabama           1.597069
Alaska            2.438038
Arizona           1.552071
Arkansas          2.284543
California        1.467011
Colorado          1.122546
Connecticut       2.452209
Delaware          1.732066
Florida           1.740771
Georgia           1.775105
Hawaii            2.553267
Idaho             3.121426
Illinois          0.816633
Indiana           1.964868
Iowa              3.405246
Kansas            2.110881
Kentucky          2.354982
Louisiana         1.298154
Maine             3.490760
Maryland          1.538992
Massachusetts     2.055379
Michigan          0.789381
Minnesota         2.962845
Mississippi       2.725416
Missouri          1.140668
Montana           2.684879
Nebraska          2.610236
Nevada            0.632824
New Hampshire     3.449005
New Jersey        1.471260
New Mexico        1.270677
New York          0.851902
North Carolina    2.947608
North Dakota      4.183835
Ohio              

## Next Steps

Congratulations on completing the assignment! Before submitting:

1. Make sure all your cells run without errors.
2. Ensure you've answered all parts of each question.
3. If any autograder tests fail, revisit your answers.

**Key Takeaways from This Lab:**

- **Unsupervised learning** works with unlabeled data to discover patterns
- **Feature scales matter**: Without standardization, large-scale features dominate distances
- **Distance metrics** (Euclidean, Manhattan) quantify similarity between observations
- **Standardization** (z-score) puts all features on equal footing
- **Distance matrices** are the foundation for clustering algorithms

In the next week, you'll use these concepts to perform dimensionality reduction
with Principal Component Analysis (PCA)!